# Phân tích Friction Score cho các task IT

Notebook hướng dẫn tính Friction Score cho các task IT và phân tích theo nhóm.

### 1. Imports và cấu hình môi trường

In [ ]:
import os
import sys
import io
import warnings
import logging
from pathlib import Path
from contextlib import redirect_stdout, redirect_stderr

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

os.environ.setdefault('KALEIDO_NO_UPDATE_CHECK', '1')
os.environ.setdefault('PLOTLY_LOG_LEVEL', 'ERROR')
os.environ.setdefault('PYTHONWARNINGS', 'ignore')

warnings.filterwarnings('ignore', category=DeprecationWarning)
for logger_name in ['plotly', 'kaleido', 'selenium', 'urllib3', 'tornado', 'choreographer']:
    logging.getLogger(logger_name).setLevel(logging.CRITICAL)
logging.disable(logging.CRITICAL)

import pandas as pd
import plotly.express as px
from src.friction_score import build_friction_outputs, build_friction_table

pd.set_option('display.max_columns', 200)

SCREENSHOT_DIR = ROOT / 'docs' / 'screenshots'
SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)

def save_chart(fig, filename, width=1200, height=700):
    output_path = SCREENSHOT_DIR / filename
    with open(os.devnull, 'w') as devnull:
        old_stdout_fd = os.dup(1)
        old_stderr_fd = os.dup(2)
        try:
            os.dup2(devnull.fileno(), 1)
            os.dup2(devnull.fileno(), 2)
            with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
                logging.disable(logging.CRITICAL)
                fig.write_image(str(output_path), width=width, height=height, scale=2)
        finally:
            logging.disable(logging.NOTSET)
            os.dup2(old_stdout_fd, 1)
            os.dup2(old_stderr_fd, 2)
            os.close(old_stdout_fd)
            os.close(old_stderr_fd)
    print(f'Đã lưu ảnh: {output_path}')
    return output_path


### 2. Chạy pipeline và kết quả

In [ ]:
INPUT_PATH = ROOT / 'data' / 'processed' / 'it_master.csv'
WORKER_PATH = ROOT / 'data' / 'processed' / 'it_worker_level.csv'

if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Chưa có dữ liệu đầu vào: {INPUT_PATH}. Hãy chạy src.data_processing trước.')

master = pd.read_csv(INPUT_PATH)
friction_table = build_friction_table(master)

# Try to read worker-level table if exists for by-group analysis, otherwise let pipeline produce it
if WORKER_PATH.exists():
    worker_level = pd.read_csv(WORKER_PATH)
else:
    worker_level = None

# Also compute by-group using function if worker_level provided
if worker_level is not None:
    from src.friction_score import build_friction_by_group
    friction_by_group = build_friction_by_group(friction_table, worker_level)
else:
    # fallback: run full pipeline (reads files internally)
    tables = build_friction_outputs()
    friction_by_group = tables.get('friction_score_by_group')

print(f'Loaded master: {INPUT_PATH} -> {master.shape[0]} rows × {master.shape[1]} cols')
print(f'Computed friction_table: {friction_table.shape[0]} rows × {friction_table.shape[1]} cols')

friction_table.head(10)


### 3. Các Task có Friction cao nhất (Top 10)

In [ ]:
top10 = friction_table.head(10).copy()

# Rút gọn tên task còn khoảng 40 ký tự
top10["Task_Short"] = top10["Task"].apply(
    lambda x: x[:40] + "..." if len(x) > 40 else x
)

fig = px.bar(
    top10,
    x="Friction Score",
    y="Task_Short",
    orientation="h",
    color="Lý do chính",
    text="Friction Score",
    hover_data=["Task"],
    title="Top 10 Task IT có Friction Score cao nhất"
)

fig.update_traces( texttemplate="%{text:.1f}", textposition="outside", cliponaxis=False)

fig.update_layout(
    template="plotly_white", width=1200, height=650, title=dict(x=0.5, font=dict(size=24)),

    xaxis=dict(
        title="Friction Score",
        range=[0,100],
        tickfont=dict(size=12)
    ),

    yaxis=dict( title="", categoryorder="total ascending", tickfont=dict(size=11), automargin=True),

    legend=dict(orientation="h", y=1.08, x=0.5, xanchor="center"),

    margin=dict(l=280, r=60, t=120, b=60)
)

save_chart(fig, 'top10_friction_score.png')
fig.show()


### Phân bố Friction Score

Biểu đồ histogram giúp kiểm tra xem điểm phân bố như thế nào (tập trung, lệch phải/trái, có nhiều giá trị 0/100 không).

In [ ]:
hist = friction_table.copy()

fig_hist = px.histogram(
    hist, 
    x='Friction Score', 
    nbins=20, 
    color='Canh_Bao',
    title='Phân bố Friction Score trên các task IT',
    labels={'Friction Score': 'Điểm Friction Score', 'count': 'Số lượng task'}
)


nguong_do = hist['Friction Score'].quantile(0.75)
fig_hist.add_vline(
    x=nguong_do, 
    line_dash='dash', 
    line_color='red',
    annotation_text=f'Ngưỡng Q75 ({nguong_do:.2f})'
)

fig_hist.update_layout(height=400)
save_chart(fig_hist, 'friction_score_histogram.png')
fig_hist.show()

### Chênh lệch AI vs Worker so với Friction Score

Scatter plot giữa `Capacity_Desire_Gap_Raw` và `Friction Score` để kiểm tra mối quan hệ trực quan giữa khoảng cách năng lực/khát vọng và lực cản.

In [ ]:
# Scatter: Capacity_Desire_Gap_Raw vs Friction Score
if 'Capacity_Desire_Gap_Raw' in friction_table.columns:
    scatter_df = friction_table.dropna(subset=['Capacity_Desire_Gap_Raw'])
    fig_scatter = px.scatter(
        scatter_df,
        x='Capacity_Desire_Gap_Raw',
        y='Friction Score',
        color='Canh_Bao',
        hover_data=['Task', 'Lý do chính'],
        trendline='ols',
        title='Capacity-Desire Gap vs Friction Score'
    )
    fig_scatter.update_layout(height=450)
    save_chart(fig_scatter, 'capacity_desire_gap_vs_friction.png')
    fig_scatter
else:
    print('Không tìm thấy cột Capacity_Desire_Gap_Raw trong bảng friction_table.')


### Top 'Lý do chính' đóng góp vào Friction

Đếm tần suất các `Lý do chính` để xem thành phần nào thường xuyên là nguyên nhân chính dẫn tới friction cao.

In [ ]:
reasons = friction_table['Lý do chính'].value_counts().reset_index()
reasons.columns = ['Lý do chính', 'Count']
fig_reasons = px.bar(reasons, x='Count', y='Lý do chính', orientation='h', text='Count', title="Top 'Lý do chính' cho Friction")
fig_reasons.update_layout(height=400)
save_chart(fig_reasons, 'top_reasons_friction.png')
fig_reasons


### 4. Phân tích theo nhóm (friction_score_by_group)

In [ ]:
friction_by_group.head(20)